# Visium Data Preprocessing (MPP=0.5 기준)

Visium 기술만 대상으로  
MPP=0.5 µm/pixel (20x) 기준 grid 패치 추출 + **3-head label** 생성

### 3-Head Label Design
- **Head A: Presence** — gene이 패치에 존재하는지 (binary, BCE loss)
- **Head B: Expression** — 존재 시 조건부 발현량 (log1p normalized, masked Gaussian NLL)
- **Head C: Uncertainty** — 예측 불확실성 (학습 시 자동 학습, label 불필요)

### Visium vs Xenium 차이점
- Visium: **spot-level** 데이터 (h5ad) — 각 spot에 좌표 + gene count matrix
- Xenium: transcript-level 데이터 (parquet) — 각 행이 개별 transcript 분자
- Gene 목록: `meaningful_marker_genes_info.csv` (1308개 meaningful genes)
- 대상 슬라이드: `visium_valid_slides.csv` (common genes 기준 필터링 완료)

In [ ]:
# ===== 1. 설정 및 라이브러리 =====
from glob import glob
import os
import numpy as np
import pandas as pd
import openslide
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import anndata
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore', message='Variable names are not unique')

def create_dir(path):
    os.makedirs(path, exist_ok=True)

# 하이퍼파라미터
TARGET_MPP = 0.5        # 20x 해상도 (µm/pixel)
CONTEXT_SCALE = 4       # 5x context = 4x wider
PATCH_SIZE = 512        # 패치 크기 (pixels)
MIN_SPOTS = 1           # 패치 내 최소 spot 수

print(f'TARGET_MPP: {TARGET_MPP} µm/pixel')
print(f'PATCH_SIZE: {PATCH_SIZE}')
print(f'CONTEXT_SCALE: {CONTEXT_SCALE} (5x)')
print(f'MIN_SPOTS: {MIN_SPOTS}')
print(f'\n3-Head Labels:')
print(f'  Head A: Presence  — binary (gene detected or not)')
print(f'  Head B: Expression — CPM normalized, per-gene p99 scaling: log1p(cpm)/log1p(p99)')
print(f'  Head C: Uncertainty — learned during training (no label needed)')

In [23]:
# ===== 2. 데이터 경로 및 Gene/Slide 목록 로드 =====
data_path = '../../data/spatialTranscriptome/'
h5ad_path = f'{data_path}st/'
save_base = '../../data/visium_marker_gene_spatial_expression'

# Marker gene 목록 로드
marker_genes_df = pd.read_csv(f'{save_base}/meaningful_marker_genes_info.csv')
MARKER_GENES = marker_genes_df['gene'].tolist()
NUM_GENES = len(MARKER_GENES)
print(f'Marker genes: {NUM_GENES}개')
print(f'처음 20개: {MARKER_GENES[:20]}')

# Valid slide 목록 로드
target_df = pd.read_csv(f'{save_base}/visium_valid_slides.csv')
print(f'\nValid slides: {len(target_df)}개')
print(target_df['st_technology'].value_counts())
print(target_df['organ'].value_counts())

# 저장 디렉토리 생성
organs = target_df['organ'].unique().tolist()
techs = target_df['st_technology'].unique().tolist()
for o in organs:
    for t in techs:
        create_dir(f'{save_base}/image/{t}/{o}')
        create_dir(f'{save_base}/image_5x/{t}/{o}')
        create_dir(f'{save_base}/label_presence/{t}/{o}')
        create_dir(f'{save_base}/label_expression/{t}/{o}')

print(f'\nOrgans: {organs}')
print('Directories created.')

Marker genes: 1308개
처음 20개: ['ACTB', 'ADAM10', 'ADAM11', 'ADAM12', 'ADAM15', 'ADAM17', 'ADAM19', 'ADAM20', 'ADAM22', 'ADAM28', 'ADAM29', 'ADAM32', 'ADAM33', 'ADAM7', 'ADAM8', 'ADAM9', 'ADAMDEC1', 'ADAMTS1', 'ADAMTS10', 'ADAMTS12']

Valid slides: 385개
st_technology
Visium          360
Visium HD        21
Visium HD 3'      4
Name: count, dtype: int64
organ
Bowel         77
Skin          63
Kidney        63
Lung          41
Brain         40
Prostate      36
Liver         14
Breast        10
Ovary          7
Bladder        6
Pancreas       6
Eye            6
Cervix         5
Lymph node     5
Lymphoid       3
Uterus         2
Heart          1
Name: count, dtype: int64

Organs: ['Ovary', 'Breast', 'Pancreas', 'Lymphoid', 'Prostate', 'Kidney', 'Lung', 'Bowel', 'Brain', 'Cervix', 'Lymph node', 'Heart', 'Skin', 'Bladder', 'Liver', 'Eye', 'Uterus']
Directories created.


In [24]:
# ===== 3. h5ad 로드 및 gene 매칭 확인 =====

# gene name → MARKER_GENES 인덱스 매핑 (빠른 조회용)
marker_gene_set = set(MARKER_GENES)
marker_gene_to_idx = {g: i for i, g in enumerate(MARKER_GENES)}

slide_gene_info = {}

print(f'{"ID":>10s} | {"Tech":>8s} | {"Organ":>12s} | {"Total Genes":>11s} | {"Matched":>10s} | {"Spots":>6s}')
print('-' * 80)

for idx in tqdm(range(len(target_df)), desc='Checking h5ad gene matching'):
    row = target_df.iloc[idx]
    sample_id = row['id']
    st_tech = row['st_technology']
    organ_name = row['organ']

    h5ad_file = f'{h5ad_path}{sample_id}.h5ad'
    if not os.path.exists(h5ad_file):
        print(f'{sample_id:>10s} | h5ad NOT FOUND')
        continue

    adata = anndata.read_h5ad(h5ad_file, backed='r')
    var_names = adata.var_names.tolist()
    n_spots = adata.n_obs

    # bytes → str
    if var_names and isinstance(var_names[0], bytes):
        var_names = [g.decode('utf-8') for g in var_names]

    var_names_set = set(var_names)
    matched = marker_gene_set & var_names_set
    missing = marker_gene_set - var_names_set

    # gene index 매핑 저장 (h5ad var_names 내 위치)
    var_name_to_idx = {g: i for i, g in enumerate(var_names)}
    gene_idx_map = {}  # marker_gene → h5ad column index
    for g in matched:
        gene_idx_map[g] = var_name_to_idx[g]

    slide_gene_info[sample_id] = {
        'gene_idx_map': gene_idx_map,
        'n_matched': len(matched),
        'n_missing': len(missing),
    }

    adata.file.close()

    print(f'{sample_id:>10s} | {st_tech:>8s} | {organ_name:>12s} | {len(var_names):>11,d} | {len(matched):>5d}/{NUM_GENES:<4d} | {n_spots:>6,d}')

print(f'\n총 {len(slide_gene_info)} / {len(target_df)} slides h5ad 확인 완료')

        ID |     Tech |        Organ | Total Genes |    Matched |  Spots
--------------------------------------------------------------------------------


Checking h5ad gene matching:   1%|          | 3/385 [00:00<00:13, 28.97it/s]

   TENX185 | Visium HD |        Ovary |      18,132 |  1308/1308 |  2,069
   TENX184 | Visium HD |        Ovary |      18,132 |  1308/1308 |  2,069
   TENX183 | Visium HD 3' |        Ovary |      38,606 |  1308/1308 |  2,113
   TENX182 | Visium HD 3' |        Ovary |      38,606 |  1308/1308 |  2,113
   TENX180 | Visium HD |       Breast |      18,085 |  1308/1308 |  2,379
   TENX179 | Visium HD 3' |     Pancreas |      38,606 |  1308/1308 |  2,164


Checking h5ad gene matching:   2%|▏         | 7/385 [00:00<00:11, 33.49it/s]

   TENX178 | Visium HD |     Lymphoid |      18,132 |  1308/1308 |  1,829
   TENX177 | Visium HD 3' |        Ovary |      38,606 |  1308/1308 |  1,963


...

    MISC12 |   Visium |        Brain |      33,538 |  1308/1308 |  4,226
    MISC11 |   Visium |        Brain |      33,538 |  1308/1308 |  4,384
    MISC10 |   Visium |        Brain |      33,538 |  1308/1308 |  4,789
     MISC9 |   Visium |        Brain |      33,538 |  1308/1308 |  4,634
     MISC8 |   Visium |        Brain |      33,538 |  1308/1308 |  3,661


Checking h5ad gene matching:  99%|█████████▉| 383/385 [00:14<00:00, 23.67it/s]

     MISC7 |   Visium |        Brain |      33,538 |  1308/1308 |  3,498
     MISC6 |   Visium |        Brain |      33,538 |  1308/1308 |  4,110
     MISC5 |   Visium |        Brain |      33,538 |  1308/1308 |  4,015
     MISC4 |   Visium |        Brain |      33,538 |  1308/1308 |  3,639
     MISC3 |   Visium |        Brain |      33,538 |  1308/1308 |  3,673


Checking h5ad gene matching: 100%|██████████| 385/385 [00:14<00:00, 26.24it/s]

     MISC2 |   Visium |        Brain |      33,538 |  1308/1308 |  3,592
     MISC1 |   Visium |        Brain |      33,538 |  1308/1308 |  3,460

총 385 / 385 slides h5ad 확인 완료


In [ ]:
# ===== 3.5. Gene별 CPM 기반 p99 통계 계산 =====
# 슬라이드별 sequencing depth 차이를 CPM으로 보정한 뒤,
# 패치 단위 gene count 분포에서 per-gene 99th percentile을 구한다.

import scipy.sparse as sp

gene_patch_values = {g: [] for g in MARKER_GENES}  # gene → list of patch-level CPM counts

for idx in tqdm(range(len(target_df)), desc='Computing per-gene stats (CPM)'):
    row = target_df.iloc[idx]
    sample_id = row['id']

    h5ad_file = f'{h5ad_path}{sample_id}.h5ad'
    if not os.path.exists(h5ad_file) or sample_id not in slide_gene_info:
        continue

    adata = anndata.read_h5ad(h5ad_file)
    var_names = adata.var_names.tolist()
    if var_names and isinstance(var_names[0], bytes):
        var_names = [g.decode('utf-8') for g in var_names]

    # spatial 좌표
    if 'spatial' in adata.obsm:
        coords = adata.obsm['spatial']
    elif 'X_spatial' in adata.obsm:
        coords = adata.obsm['X_spatial']
    elif 'pxl_col_in_fullres' in adata.obs.columns:
        coords = adata.obs[['pxl_col_in_fullres', 'pxl_row_in_fullres']].values
    elif 'imagecol' in adata.obs.columns:
        coords = adata.obs[['imagecol', 'imagerow']].values
    else:
        continue

    spot_x = coords[:, 0].astype(np.float64)
    spot_y = coords[:, 1].astype(np.float64)

    # count matrix → CPM 정규화
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float64)

    total_per_spot = X.sum(axis=1, keepdims=True)
    total_per_spot = np.where(total_per_spot == 0, 1.0, total_per_spot)
    X_cpm = X / total_per_spot * 10000  # Counts Per 10k

    # marker gene column indices
    var_name_to_idx = {g: i for i, g in enumerate(var_names)}
    marker_col_indices = np.array([var_name_to_idx.get(g, -1) for g in MARKER_GENES])
    valid_mask = marker_col_indices >= 0

    marker_cpm = np.zeros((X_cpm.shape[0], NUM_GENES), dtype=np.float64)
    for gi in range(NUM_GENES):
        if valid_mask[gi]:
            marker_cpm[:, gi] = X_cpm[:, marker_col_indices[gi]]

    # WSI MPP → grid 좌표 변환
    image_file = row['image_filename']
    wsi_path = f'{data_path}wsis/{image_file}'
    if not os.path.exists(wsi_path):
        continue

    slide = openslide.open_slide(wsi_path)
    slide_w, slide_h = slide.dimensions
    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0))
    if slide_mpp <= 0:
        slide.close()
        continue
    if slide_mpp < 0.1:
        slide_mpp *= 10

    downsample_20x = TARGET_MPP / slide_mpp
    spot_x_20x = spot_x / downsample_20x
    spot_y_20x = spot_y / downsample_20x

    n_patches_x = int(slide_w / downsample_20x) // PATCH_SIZE
    n_patches_y = int(slide_h / downsample_20x) // PATCH_SIZE
    slide.close()

    # 패치별 gene count 합산 → 분포 수집
    for py in range(n_patches_y):
        for px in range(n_patches_x):
            x0, y0 = px * PATCH_SIZE, py * PATCH_SIZE
            in_patch = (
                (spot_x_20x >= x0) & (spot_x_20x < x0 + PATCH_SIZE) &
                (spot_y_20x >= y0) & (spot_y_20x < y0 + PATCH_SIZE)
            )
            if in_patch.sum() < MIN_SPOTS:
                continue
            gene_counts = marker_cpm[in_patch].sum(axis=0)
            for gi in range(NUM_GENES):
                if gene_counts[gi] > 0:
                    gene_patch_values[MARKER_GENES[gi]].append(gene_counts[gi])

# gene별 p99 계산
gene_p99 = {}
for g in MARKER_GENES:
    vals = gene_patch_values[g]
    if len(vals) > 0:
        gene_p99[g] = np.percentile(vals, 99)
    else:
        gene_p99[g] = 1.0  # fallback

# CSV 저장
p99_df = pd.DataFrame({
    'gene': MARKER_GENES,
    'p99_cpm': [gene_p99[g] for g in MARKER_GENES],
    'n_patches': [len(gene_patch_values[g]) for g in MARKER_GENES],
    'log1p_p99': [np.log1p(gene_p99[g]) for g in MARKER_GENES],
})
p99_csv_path = f'{save_base}/visium_gene_p99_cpm.csv'
p99_df.to_csv(p99_csv_path, index=False)

print(f'Gene p99 통계 저장: {p99_csv_path}')
print(f'p99 범위: {p99_df["p99_cpm"].min():.2f} ~ {p99_df["p99_cpm"].max():.2f}')
print(f'패치 수 범위: {p99_df["n_patches"].min()} ~ {p99_df["n_patches"].max()}')
print(p99_df.head(20))

In [ ]:
# ===== 4. Grid 패치 추출 + 3-head label (Presence + Expression with per-gene CPM normalization) =====
from concurrent.futures import ThreadPoolExecutor
import glob as glob_module

save_image_dir = f'{save_base}/image'
save_image_5x_dir = f'{save_base}/image_5x'
save_presence_dir = f'{save_base}/label_presence'
save_expression_dir = f'{save_base}/label_expression'

# gene별 log1p(p99) 배열 (NUM_GENES,) — per-gene normalization 기준
log1p_p99 = np.array([np.log1p(gene_p99[g]) for g in MARKER_GENES], dtype=np.float32)
# 0 방지 (발현 없는 gene fallback)
log1p_p99 = np.where(log1p_p99 == 0, 1.0, log1p_p99)

print(f'Per-gene p99 normalization 적용')
print(f'log1p(p99) 범위: {log1p_p99.min():.4f} ~ {log1p_p99.max():.4f}')

# 이미 처리된 슬라이드 스킵
existing_files = set()
for f in glob_module.glob(f'{save_image_dir}/**/*.tiff', recursive=True):
    basename = os.path.splitext(os.path.basename(f))[0]
    parts = basename.rsplit('_', 2)
    if len(parts) == 3:
        existing_files.add(parts[0])
print(f'Already processed slides: {len(existing_files)}')

skipped = []
processed = []

for idx in tqdm(range(len(target_df)), desc='Grid extraction (Visium h5ad)'):
    row = target_df.iloc[idx]
    sample_id = row['id']
    organ_name = row['organ']
    st_tech = row['st_technology']
    image_file = row['image_filename']
    patch_count = 0

    if sample_id in existing_files:
        continue

    # h5ad 존재 확인
    h5ad_file = f'{h5ad_path}{sample_id}.h5ad'
    if not os.path.exists(h5ad_file):
        skipped.append((sample_id, 'h5ad not found'))
        continue

    if sample_id not in slide_gene_info:
        skipped.append((sample_id, 'gene info not available'))
        continue

    # h5ad 로드
    adata = anndata.read_h5ad(h5ad_file)
    var_names = adata.var_names.tolist()
    if var_names and isinstance(var_names[0], bytes):
        var_names = [g.decode('utf-8') for g in var_names]

    # spatial 좌표 추출
    if 'spatial' in adata.obsm:
        coords = adata.obsm['spatial']
    elif 'X_spatial' in adata.obsm:
        coords = adata.obsm['X_spatial']
    else:
        coord_cols = [c for c in adata.obs.columns if c in ['x', 'y', 'pxl_col_in_fullres', 'pxl_row_in_fullres',
                                                              'array_col', 'array_row', 'imagecol', 'imagerow']]
        if 'pxl_col_in_fullres' in adata.obs.columns and 'pxl_row_in_fullres' in adata.obs.columns:
            coords = adata.obs[['pxl_col_in_fullres', 'pxl_row_in_fullres']].values
        elif 'imagecol' in adata.obs.columns and 'imagerow' in adata.obs.columns:
            coords = adata.obs[['imagecol', 'imagerow']].values
        else:
            skipped.append((sample_id, f'spatial coords not found, obsm keys: {list(adata.obsm.keys())}, obs cols: {list(adata.obs.columns)[:10]}'))
            continue

    spot_x = coords[:, 0].astype(np.float64)  # level0 pixel x
    spot_y = coords[:, 1].astype(np.float64)  # level0 pixel y

    # count matrix → CPM 정규화
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float64)

    total_per_spot = X.sum(axis=1, keepdims=True)
    total_per_spot = np.where(total_per_spot == 0, 1.0, total_per_spot)
    X_cpm = (X / total_per_spot * 10000).astype(np.float32)

    # marker gene 인덱스 매핑
    var_name_to_idx = {g: i for i, g in enumerate(var_names)}
    marker_col_indices = []
    for g in MARKER_GENES:
        if g in var_name_to_idx:
            marker_col_indices.append(var_name_to_idx[g])
        else:
            marker_col_indices.append(-1)
    marker_col_indices = np.array(marker_col_indices)

    # marker gene CPM matrix (n_spots × NUM_GENES)
    marker_cpm = np.zeros((X_cpm.shape[0], NUM_GENES), dtype=np.float32)
    valid_mask = marker_col_indices >= 0
    for gi in range(NUM_GENES):
        if valid_mask[gi]:
            marker_cpm[:, gi] = X_cpm[:, marker_col_indices[gi]]

    # WSI 로드
    wsi_path = f'{data_path}wsis/{image_file}'
    if not os.path.exists(wsi_path):
        skipped.append((sample_id, 'WSI not found'))
        continue

    slide = openslide.open_slide(wsi_path)
    slide_w, slide_h = slide.dimensions

    # MPP
    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0))
    if slide_mpp <= 0:
        skipped.append((sample_id, 'MPP not found in image header'))
        slide.close()
        continue
    if slide_mpp < 0.1:
        slide_mpp *= 10

    downsample_20x = TARGET_MPP / slide_mpp
    downsample_5x = downsample_20x * CONTEXT_SCALE

    full_patch_20x = int(PATCH_SIZE * downsample_20x)
    full_patch_5x = int(PATCH_SIZE * downsample_5x)

    target_w = int(slide_w / downsample_20x)
    target_h = int(slide_h / downsample_20x)
    n_patches_x = target_w // PATCH_SIZE
    n_patches_y = target_h // PATCH_SIZE

    if n_patches_x == 0 or n_patches_y == 0:
        skipped.append((sample_id, f'slide too small: {slide_w}x{slide_h}, mpp={slide_mpp:.4f}'))
        slide.close()
        continue

    # spot 좌표를 20x pixel 좌표로 변환
    spot_x_20x = spot_x / downsample_20x
    spot_y_20x = spot_y / downsample_20x

    # 슬라이드별 저장 디렉토리 생성
    create_dir(f'{save_image_dir}/{st_tech}/{organ_name}/{sample_id}')
    create_dir(f'{save_image_5x_dir}/{st_tech}/{organ_name}/{sample_id}')
    create_dir(f'{save_presence_dir}/{st_tech}/{organ_name}/{sample_id}')
    create_dir(f'{save_expression_dir}/{st_tech}/{organ_name}/{sample_id}')

    def save_patch(px, py):
        patch_x0 = px * PATCH_SIZE
        patch_y0 = py * PATCH_SIZE

        # 패치 내 spot 찾기
        in_patch = (
            (spot_x_20x >= patch_x0) & (spot_x_20x < patch_x0 + PATCH_SIZE) &
            (spot_y_20x >= patch_y0) & (spot_y_20x < patch_y0 + PATCH_SIZE)
        )
        n_spots_in_patch = in_patch.sum()
        if n_spots_in_patch < MIN_SPOTS:
            return False

        # 패치 내 spot들의 marker gene CPM count 합산
        gene_counts_cpm = marker_cpm[in_patch].sum(axis=0)  # (NUM_GENES,)

        # Head A: Presence (binary — gene이 패치에 존재하는지)
        label_presence = (gene_counts_cpm > 0).astype(np.float32)

        # Head B: Expression (per-gene CPM normalized — 조건부 발현량)
        # log1p(cpm_count) / log1p(p99) → gene별 독립 스케일링
        label_expression = np.clip(np.log1p(gene_counts_cpm) / log1p_p99, 0.0, 1.0).astype(np.float32)

        # 20x patch
        full_x = int(px * full_patch_20x)
        full_y = int(py * full_patch_20x)
        img_region = slide.read_region((full_x, full_y), 0, (full_patch_20x, full_patch_20x))
        img_patch = img_region.convert('RGB').resize((PATCH_SIZE, PATCH_SIZE), Image.LANCZOS)

        # 5x context patch
        center_x = full_x + full_patch_20x // 2
        center_y = full_y + full_patch_20x // 2
        half_5x = full_patch_5x // 2
        x5 = int(np.clip(center_x - half_5x, 0, max(0, slide_w - full_patch_5x)))
        y5 = int(np.clip(center_y - half_5x, 0, max(0, slide_h - full_patch_5x)))
        img_region_5x = slide.read_region((x5, y5), 0, (full_patch_5x, full_patch_5x))
        img_patch_5x = img_region_5x.convert('RGB').resize((PATCH_SIZE, PATCH_SIZE), Image.LANCZOS)

        patch_name = f'{sample_id}_{px}_{py}'
        img_patch.save(f'{save_image_dir}/{st_tech}/{organ_name}/{sample_id}/{patch_name}.tiff')
        img_patch_5x.save(f'{save_image_5x_dir}/{st_tech}/{organ_name}/{sample_id}/{patch_name}.tiff')
        np.save(f'{save_presence_dir}/{st_tech}/{organ_name}/{sample_id}/{patch_name}.npy', label_presence)
        np.save(f'{save_expression_dir}/{st_tech}/{organ_name}/{sample_id}/{patch_name}.npy', label_expression)
        return True

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for py in range(n_patches_y):
            for px in range(n_patches_x):
                futures.append(executor.submit(save_patch, px, py))
        for f in futures:
            if f.result():
                patch_count += 1

    slide.close()
    processed.append((sample_id, patch_count))
    print(f'{sample_id} | {st_tech}/{organ_name} | MPP={slide_mpp:.4f} | ds={downsample_20x:.2f} | {patch_count} patches | {len(spot_x)} spots')

print(f'\n=== Done ===')
print(f'Processed: {len(processed)} slides')
print(f'Skipped: {len(skipped)} slides')
for s_id, reason in skipped:
    print(f'  SKIP {s_id}: {reason}')

In [ ]:
# ===== 5. 처리 결과 요약 =====
if processed:
    total_patches = sum(cnt for _, cnt in processed)
    print(f'Total processed slides: {len(processed)}')
    print(f'Total patches saved: {total_patches}')
    print(f'Average patches per slide: {total_patches / len(processed):.1f}')
    print(f'\nTop 10 slides by patch count:')
    for s_id, cnt in sorted(processed, key=lambda x: x[1], reverse=True)[:10]:
        print(f'  {s_id}: {cnt} patches')

for tech in target_df['st_technology'].unique():
    n_files = len(glob(f'{save_base}/image/{tech}/**/*.tiff', recursive=True))
    print(f'\n{tech}: {n_files} image patches saved')

In [ ]:
# ===== 6. 시각화: 샘플 슬라이드 확인 (Presence + Expression) =====
from matplotlib.colors import Normalize
from matplotlib import cm

def visualize_slide_visium(meta_row, data_path, h5ad_dir, marker_genes, marker_gene_to_idx,
                           target_mpp=0.5, patch_size=512, thumb_long_side=2000):
    sample_id = meta_row['id']
    image_file = meta_row['image_filename']

    # h5ad 로드
    h5ad_file = f'{h5ad_dir}{sample_id}.h5ad'
    adata = anndata.read_h5ad(h5ad_file)
    var_names = adata.var_names.tolist()
    if var_names and isinstance(var_names[0], bytes):
        var_names = [g.decode('utf-8') for g in var_names]

    # spatial 좌표
    if 'spatial' in adata.obsm:
        coords = adata.obsm['spatial']
    elif 'X_spatial' in adata.obsm:
        coords = adata.obsm['X_spatial']
    else:
        if 'pxl_col_in_fullres' in adata.obs.columns:
            coords = adata.obs[['pxl_col_in_fullres', 'pxl_row_in_fullres']].values
        elif 'imagecol' in adata.obs.columns:
            coords = adata.obs[['imagecol', 'imagerow']].values
        else:
            print(f'{sample_id}: spatial coords not found')
            return

    spot_x = coords[:, 0].astype(np.float64)
    spot_y = coords[:, 1].astype(np.float64)

    # count matrix
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    var_name_to_idx_local = {g: i for i, g in enumerate(var_names)}

    slide = openslide.open_slide(f'{data_path}wsis/{image_file}')
    slide_w, slide_h = slide.dimensions

    slide_mpp = float(slide.properties.get('openslide.mpp-x', 0))
    if slide_mpp <= 0:
        print(f'{sample_id}: MPP not found')
        return
    if slide_mpp < 0.1:
        slide_mpp *= 10
    downsample_20x = target_mpp / slide_mpp

    aspect = slide_w / slide_h
    if aspect >= 1:
        tw, th = thumb_long_side, int(thumb_long_side / aspect)
    else:
        tw, th = int(thumb_long_side * aspect), thumb_long_side
    thumbnail = slide.get_thumbnail((tw, th))
    tw, th = thumbnail.size
    scale_x = tw / slide_w
    scale_y = th / slide_h

    spot_x_20x = spot_x / downsample_20x
    spot_y_20x = spot_y / downsample_20x
    full_patch_20x = int(patch_size * downsample_20x)
    n_patches_x = int(slide_w / downsample_20x) // patch_size
    n_patches_y = int(slide_h / downsample_20x) // patch_size

    # 패치별 정보 수집
    patch_info = {}
    for py in range(n_patches_y):
        for px in range(n_patches_x):
            x0, y0 = px * patch_size, py * patch_size
            in_patch = (
                (spot_x_20x >= x0) & (spot_x_20x < x0 + patch_size) &
                (spot_y_20x >= y0) & (spot_y_20x < y0 + patch_size)
            )
            n_spots = in_patch.sum()
            if n_spots < 1:
                continue
            spot_counts = X[in_patch]
            patch_info[(px, py)] = {
                'n_spots': n_spots,
                'spot_counts': spot_counts,
            }

    # 시각화할 gene 선택 (처음 6개)
    vis_genes = marker_genes[:6]
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'{sample_id} — Visium Spot Expression', fontsize=14)

    for gi, gene in enumerate(vis_genes):
        ax = axes[gi // 3, gi % 3]
        ax.imshow(thumbnail, alpha=0.7)
        ax.set_title(gene, fontsize=11)

        if gene not in var_name_to_idx_local:
            ax.set_title(f'{gene} (not found)')
            continue

        gene_col = var_name_to_idx_local[gene]
        values = []
        centers_x = []
        centers_y = []

        for (px, py), info in patch_info.items():
            gene_count = info['spot_counts'][:, gene_col].sum()
            if gene_count > 0:
                cx = (px * patch_size + patch_size / 2) * downsample_20x * scale_x
                cy = (py * patch_size + patch_size / 2) * downsample_20x * scale_y
                centers_x.append(cx)
                centers_y.append(cy)
                values.append(np.log1p(gene_count))

        if values:
            norm = Normalize(vmin=0, vmax=max(values))
            colors = cm.hot(norm(np.array(values)))
            ax.scatter(centers_x, centers_y, c=colors, s=15, alpha=0.8, edgecolors='none')

        ax.axis('off')

    plt.tight_layout()
    plt.show()
    slide.close()

# 첫 번째 유효 슬라이드로 시각화
if len(target_df) > 0:
    sample_row = target_df.iloc[0]
    visualize_slide_visium(sample_row, data_path, h5ad_path, MARKER_GENES, marker_gene_to_idx)